# Amber Molecular Dynamics User Guide

**Version:** 1.0.0.  
**Created:** 2026-04-08.  
**Last Updated:** 2026-04-08.  
**Tested on:** 2026-04-08.  

**Audience:** Computational chemists and biophysicists running protein simulations on HPC systems

**System:** Spack-managed HPC cluster with CPU/GPU variants and automated MPI/UCX workflow

## TL;DR

This guide explains **how to run Amber molecular dynamics simulations on this HPC system**.

**What is Amber?** A biomolecular simulation suite for running molecular dynamics on proteins, nucleic acids, and small molecules — from energy minimization to microsecond production runs.

Run one simple case first:

```bash
# 1. Load both modules
module load amber ambertools

# 2. Verify the key tools are in PATH
tleap -h && sander -h

# 3. Check your GPU engine (on a GPU node)
pmemd.cuda -h
```

It covers:
- Quick setup and first simulation
- CPU vs GPU decision-making for Amber
- Complete `sander.MPI` and `pmemd.cuda` workflow templates
- Step-by-step troubleshooting

The goal is to **get from zero to a running simulation in 15 minutes**, then optimize for production runs.

**Key pipeline:** `tleap` → `sander` (minimize) → `sander.MPI` / `pmemd.cuda` (heat + produce) → `cpptraj` (analyze)

## Table of Contents

- [TL;DR](#tldr)
- [PART I — Getting Started (Read Once)](#part-i--getting-started-read-once)
  - [TL;DR: 5-Minute Quick Start](#tldr-5-minute-quick-start)
  - [Step 1: Load the Modules](#step-1-load-the-modules)
  - [Step 2: Verify Setup](#step-2-verify-setup)
  - [Step 3: Run Your First Job](#step-3-run-your-first-job)
- [PART II — How to Use Amber (Read Carefully Once)](#part-ii--how-to-use-amber-read-carefully-once)
  - [What You Need to Know](#what-you-need-to-know)
  - [CPU vs GPU: When to Use Each](#cpu-vs-gpu-when-to-use-each)
  - [Resource Requests](#resource-requests)
- [PART III — Workflows (Use As Needed)](#part-iii--workflows-use-as-needed)
  - [Configuration Variables](#configuration-variables)
  - [Input File Setup](#input-file-setup)
  - [Workflow 1: Single-Node CPU Simulation](#workflow-1-single-node-cpu-simulation)
  - [Workflow 2: GPU-Accelerated Simulation](#workflow-2-gpu-accelerated-simulation)
  - [Workflow 3: Multi-Node Production Run](#workflow-3-multi-node-production-run)
- [PART IV — Troubleshooting (Skim When Broken)](#part-iv--troubleshooting-skim-when-broken)
  - [Monitoring Your Job](#monitoring-your-job)
  - [Symptom-Based Lookup](#symptom-based-lookup)
  - [Understanding Output Files](#understanding-output-files)
  - [FAQ](#faq)
  - [External Resources](#external-resources)

_💡 Tip: Enable nbextensions like "Collapsible Headings" and "Table of Contents (2)" for better navigation._

---

## PART I — Getting Started (Read Once)

**Purpose:** "I just want this to work."

This part gets you from zero to a running Amber simulation in minimal time. No deep understanding needed yet—just follow the steps.

### TL;DR: 5-Minute Quick Start

If you just want to run a simulation:

```bash
# 1. Load both modules
module load amber ambertools

# 2. Verify tools are available
tleap -h

# 3. Prep system and submit
tleap -f leaprc.input
sbatch amber.sbatch
```

**Expected result:** Job submits; `.mdout`, `.rst7`, and `.nc` files appear in your working directory when complete.

---

### Step 1: Load the Modules

All software is pre-installed and managed through the **module system** (lmod). Amber is split across two modules:

- `ambertools` — `tleap`, `sander`, `sander.MPI`, `antechamber`, `cpptraj` (free)
- `amber` — `pmemd.cuda`, `pmemd.MPI.cuda` (licensed GPU engines)

In [ ]:
%%bash
# List available versions (optional)
module avail amber ambertools

**Expected output:**
```
amber/25    ambertools/25 (default)
```

Now load both modules:

In [ ]:
%%bash
# Load both modules (required for all workflows)
module load amber ambertools

**Expected output:** No output on success. (Use Step 2 below to verify.)

---

### Step 2: Verify Setup

Confirm both modules loaded correctly and all key tools are in your PATH:

In [ ]:
%%bash
# Load and verify all key tools are available
module load amber ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -3

echo ""
echo "=== Core tools ==="
for tool in tleap sander sander.MPI antechamber cpptraj pmemd.cuda; do
    command -v $tool &>/dev/null && echo "  OK  $tool" || echo "  --  $tool (not found)"
done

**Expected output:**
```
Python 3.11.x :: Amber distribution

=== Core tools ===
  OK  tleap
  OK  sander
  OK  sander.MPI
  OK  antechamber
  OK  cpptraj
  OK  pmemd.cuda
```

If `tleap` or `sander` show `not found`, run `module load ambertools` before retrying.  
If `pmemd.cuda` shows `not found`, run `module load amber` on a GPU node.

✅ **Success!** Your environment is ready.

---

### Step 3: Run Your First Job

Create a minimal alanine PDB — the canonical Amber test structure — and submit a test energy-minimization job to confirm everything works end-to-end:

In [ ]:
%%bash
# Create a minimal single-residue alanine PDB (Amber standard test structure)
mkdir -p input

cat > input/ala.pdb << 'EOF'
ATOM      1  N   ALA A   1       1.201   0.847   0.000  1.00  0.00           N
ATOM      2  CA  ALA A   1       0.000   0.000   0.000  1.00  0.00           C
ATOM      3  C   ALA A   1      -1.250   0.880   0.000  1.00  0.00           C
ATOM      4  O   ALA A   1      -1.170   2.090  -0.003  1.00  0.00           O
ATOM      5  CB  ALA A   1       0.073  -0.774   1.303  1.00  0.00           C
ATOM      6  H   ALA A   1       1.200   1.431   0.818  1.00  0.00           H
ATOM      7  HA  ALA A   1       0.052  -0.602  -0.905  1.00  0.00           H
ATOM      8  HB1 ALA A   1       1.091  -1.167   1.401  1.00  0.00           H
ATOM      9  HB2 ALA A   1      -0.596  -1.611   1.302  1.00  0.00           H
ATOM     10  HB3 ALA A   1      -0.215  -0.125   2.152  1.00  0.00           H
TER
END
EOF

echo "Created test PDB:"
ls -lah input/ala.pdb

**Expected output:**
```
Created test PDB:
-rw-r--r-- 1 user group 782 Apr  8 10:00 input/ala.pdb
```

In [ ]:
%%bash
# Generate input files and submit a test minimization job
module load amber ambertools 2>/dev/null

mkdir -p mdin_files

cat > mdin_files/em_quick.mdin << 'EOF'
Amber quick energy minimization (Part I test)
 &cntrl
  imin   = 1,
  maxcyc = 5000,
  ncyc   = 2500,
  ntb    = 1,
  ntp    = 0,
  cut    = 8.0,
  ntpr   = 100,
  ntwr   = 5000,
 /
EOF

cat > leaprc_quick.input << 'EOF'
source leaprc.protein.ff99SBildn
source leaprc.water.spce
mol = loadpdb input/ala.pdb
check mol
solvateBox mol SPCBOX 10.0
addions mol Na+ 0
saveamberparm mol system.prmtop system.inpcrd
savepdb mol system.pdb
quit
EOF

cat > amber_test.sbatch << 'ENDJOB'
#!/bin/bash
#SBATCH --job-name=amber-test-em
#SBATCH --partition=build
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --mem=8G
#SBATCH --time=00:30:00
#SBATCH --output=amber_test_%j.out
#SBATCH --error=amber_test_%j.err
module load amber ambertools
tleap -f leaprc_quick.input
sander -O \
  -i mdin_files/em_quick.mdin -o em_test.mdout \
  -p system.prmtop -c system.inpcrd -r em_test.rst7
echo "✅ Test minimization complete."
ENDJOB

echo "=== Submitting test minimization job ==="
sbatch amber_test.sbatch

**Expected output:**
```
=== Submitting test minimization job ===
Submitted batch job 12345
```

Check status:

In [ ]:
%%bash
echo "=== Your Amber Jobs ==="
squeue -u $USER --format="%.7i %.9P %.20j %.8u %.2t %.10M %.6D %R"

**Expected output:**
```
  JOBID PARTITION                 NAME     USER  ST       TIME  NODES
  12345     build        amber-test-em   user   R       0:30      1
```

**Looking for:** `ST=R` (Running) or `ST=PD` (Pending — waiting for resources). If not listed, the job already completed — check `sacct -u $USER`.

🎉 **You're set up!** Continue to Part II to understand resource decisions, or jump to Part III for production workflows.

**Next Steps:**
- **Want to understand sander vs pmemd and when to use GPU?** Continue to [Part II](#part-ii--how-to-use-amber-read-carefully-once).
- **Ready for production workflows?** Jump to [Part III](#part-iii--workflows-use-as-needed).
- **Hit an error?** Go to [Part IV Troubleshooting](#part-iv--troubleshooting-skim-when-broken).

---

## PART II — How to Use Amber (Read Carefully Once)

**Purpose:** "Help me not screw this up."

This part explains key concepts and decision-making. You'll understand **when** to use different options and **why** certain choices matter.

### What You Need to Know

#### The Module Split

Amber's tools are split across two modules — both are needed for every workflow:

| Module | Tools | Purpose |
|--------|-------|---------|
| `ambertools` | `tleap`, `sander`, `sander.MPI`, `antechamber`, `cpptraj` | Free tools — system prep, CPU MD, analysis |
| `amber` | `pmemd.cuda`, `pmemd.MPI.cuda` | Licensed — GPU-accelerated MD only |

Always load both: `module load amber ambertools`.

#### The Simulation Pipeline

Every Amber simulation follows this sequence:

| Stage | Tool | Input → Output |
|-------|------|----------------|
| System preparation | `tleap` | `.pdb` → `.prmtop` + `.inpcrd` |
| Energy minimization | `sander` | `.inpcrd` → `.rst7` |
| Heating (NVT, 0→300 K) | `sander.MPI` or `pmemd.cuda` | `.rst7` → `.rst7` |
| Production (NPT, 300 K) | `sander.MPI` or `pmemd.cuda` | `.rst7` → `.nc` trajectory |
| Analysis | `cpptraj` | `.nc` → RMSD, clustering, etc. |

**Rule:** Always run energy minimization with `sander` (serial, most robust), then switch to `pmemd.cuda` for heating and production if a GPU is available.

### CPU vs GPU: When to Use Each

**Use CPU (`sander.MPI`) if:**
- Your system is small (< 5K atoms)
- You're testing or debugging
- GPU nodes are unavailable

**Use GPU (`pmemd.cuda`) if:**
- Your system is > 5K atoms
- You're running production simulations
- You need results quickly (50–200× faster than CPU)

#### Decision Guide

```
How many atoms in your system?
│
├─ < 5,000 atoms
│   └─ CPU only  (GPU overhead dominates; use sander.MPI)
│
├─ 5,000 – 100,000 atoms
│   ├─ GPU available? → pmemd.cuda  (50–200x faster)
│   └─ No GPU?       → sander.MPI, 16–32 ranks
│
└─ > 100,000 atoms
    ├─ GPU available? → pmemd.cuda (single) or pmemd.MPI.cuda (multi-GPU)
    └─ No GPU?       → multi-node sander.MPI  (Workflow 3)
```

### Resource Requests

**Don't request more than you need.**

| Partition | System Size | CPU Ranks | GPU | Typical Request | Performance |
|-----------|---|---|---|---|---|
| **build** | < 10K atoms | 4–16 | ❌ | `--ntasks=8` | ~5 ns/day |
| **build** | 10–50K atoms | 16–32 | ❌ | `--ntasks=16` | ~2 ns/day |
| **gpu** | 5–50K atoms | 1–4 | ✅ 1× | `--ntasks=1 --gres=gpu:1` | ~200 ns/day |
| **gpu** | 50–100K atoms | 1–4 | ✅ 1× | `--ntasks=1 --gres=gpu:1` | ~50 ns/day |
| **gpu** | > 100K atoms | 4–8 | ✅ 2× | `--ntasks=2 --gres=gpu:2` | ~20 ns/day |

For questions about resources, see Part IV FAQ.

---

## PART III — Workflows (Use As Needed)

**Purpose:** "Show me how to actually do science."

Pick the workflow that matches your task. Each is self-contained.

### Configuration Variables

Set these **once** before running any workflow below:

In [ ]:
%%bash
# Set environment variables for all Amber workflows
# Customize these for your project

export PROJECT_NAME="my_amber_project"
export INPUT_PDB="ala.pdb"
export WORK_DIR="${PWD}/amber_work"
export MPI_RANKS=16
export FORCE_FIELD="ff99SBildn"
export WATER_MODEL="spce"
export USE_GPU="false"
export GPU_PARTITION="gpu"
export CPU_PARTITION="build"

echo "=== Amber Workflow Configuration ==="
echo "Project:           $PROJECT_NAME"
echo "Input structure:   $INPUT_PDB"
echo "Work directory:    $WORK_DIR"
echo "MPI ranks:         $MPI_RANKS"
echo "Force field:       leaprc.protein.$FORCE_FIELD"
echo "Water model:       leaprc.water.$WATER_MODEL"
echo "Use GPU:           $USE_GPU"
echo ""
echo "✅ Configuration loaded. Use these variables in templates below."

### Input File Setup

All workflows share the same `leaprc.input` and three `.mdin` files. Create them here:

In [ ]:
import os

# Energy minimization control file
em_mdin = """Amber energy minimization
 &cntrl
  imin   = 1,          ! Perform minimization (not MD)
  maxcyc = 5000,       ! Max minimization steps
  ncyc   = 2500,       ! Steepest descent before switching to CG
  ntb    = 1,          ! Periodic boundary (constant volume)
  ntp    = 0,
  cut    = 8.0,
  ntpr   = 100,
  ntwr   = 5000,
 /
"""

# Heating control file: NVT, 0 → 300 K, 50 ps
heat_mdin = """Amber NVT heating 0 to 300 K (50 ps)
 &cntrl
  imin   = 0,
  irest  = 0,  ntx = 1,
  nstlim = 25000,  dt = 0.002,
  ntb    = 1,  ntp = 0,
  ntt    = 3,  gamma_ln = 1.0,
  tempi  = 0.0,  temp0 = 300.0,
  ntpr   = 100,  ntwx = 100,  ntwr = 5000,
  cut    = 8.0,
  nmropt = 1,
 /
 &wt TYPE='TEMP0', istep1=0, istep2=25000, value1=0.0, value2=300.0, /
 &wt TYPE='END', /
"""

# Production MD control file: NPT, 300 K / 1 bar, 100 ps
prod_mdin = """Amber NPT production MD (100 ps at 300 K / 1 bar)
 &cntrl
  imin   = 0,
  irest  = 1,  ntx = 5,
  nstlim = 50000,  dt = 0.002,
  ntb    = 2,  ntp = 1,
  pres0  = 1.0,  taup = 2.0,
  ntt    = 3,  gamma_ln = 1.0,
  temp0  = 300.0,
  ntpr   = 100,  ntwx = 100,  ntwr = 5000,
  cut    = 8.0,
  iwrap  = 1,
 /
"""

# tleap script: ff99SBildn force field, SPC/E water
leaprc = """source leaprc.protein.ff99SBildn
source leaprc.water.spce
mol = loadpdb input/ala.pdb
check mol
solvateBox mol SPCBOX 10.0
addions mol Na+ 0
saveamberparm mol system.prmtop system.inpcrd
savepdb mol system.pdb
quit
"""

os.makedirs('mdin_files', exist_ok=True)

with open('mdin_files/em.mdin', 'w') as f:    f.write(em_mdin)
with open('mdin_files/heat.mdin', 'w') as f:  f.write(heat_mdin)
with open('mdin_files/prod.mdin', 'w') as f:  f.write(prod_mdin)
with open('leaprc.input', 'w') as f:           f.write(leaprc)

print("✅ Created Amber input files:")
print("  • mdin_files/em.mdin    (energy minimization, 5,000 steps)")
print("  • mdin_files/heat.mdin  (NVT heating 0→300 K, 50 ps)")
print("  • mdin_files/prod.mdin  (NPT production MD, 100 ps)")
print("  • leaprc.input          (ff99SBildn + SPC/E water)")

**Expected output:**
```
✅ Created Amber input files:
  • mdin_files/em.mdin    (energy minimization, 5,000 steps)
  • mdin_files/heat.mdin  (NVT heating 0→300 K, 50 ps)
  • mdin_files/prod.mdin  (NPT production MD, 100 ps)
  • leaprc.input          (ff99SBildn + SPC/E water)
```

---

### Workflow 1: Single-Node CPU Simulation

[Step-by-step: system < 50K atoms, CPU partition, quick turnaround needed]

In [ ]:
%%bash
# WORKFLOW 1: Single-Node CPU Simulation
# Runs tleap → sander (EM) → sander.MPI (heat) → sander.MPI (prod)
# Script: amber.sbatch  (loops over all PDB files in INPUT_DIR)

module load amber ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -2

echo ""
echo "=== Running tleap to build system ==="
tleap -f leaprc.input 2>&1 | tail -12

echo ""
echo "=== System files ready ==="
ls -lh system.prmtop system.inpcrd system.pdb 2>/dev/null || echo "NOTE: tleap not available — check: module load ambertools"

# PRODUCTION SUBMISSION (uncomment when ready)
# ⚠️ Warning: submits a multi-hour job to the build partition via amber.sbatch
#
# INPUT_DIR=./input OUTPUT_BASE=./output MDIN_DIR=./mdin_files LEAPRC=./leaprc.input \
# sbatch --partition=build --nodes=1 --ntasks=16 amber.sbatch

**Expected output:**
```
Welcome to LEaP, version 24.x ...
...
Exiting LEaP: Errors = 0; Warnings = 0; Notes = 1.

=== System files ready ===
-rw-r--r-- 1 user group 310K Apr  8 10:01 system.prmtop
-rw-r--r-- 1 user group  52K Apr  8 10:01 system.inpcrd
-rw-r--r-- 1 user group  21K Apr  8 10:01 system.pdb
```

Submit using `amber.sbatch`:
```bash
INPUT_DIR=./input OUTPUT_BASE=./output MDIN_DIR=./mdin_files LEAPRC=./leaprc.input \
sbatch --partition=build --nodes=1 --ntasks=16 amber.sbatch
```

`amber.sbatch` loops over every `.pdb` in `INPUT_DIR`, runs `tleap → sander (EM) → sander.MPI (heat + prod)` for each, and saves results under `OUTPUT_BASE/<basename>/`.

**⚠️ Before submitting:**
1. Run `tleap -f leaprc.input` on the login node first to confirm `Errors = 0`
2. Adjust `--ntasks` to match your system size (16 ranks is the practical ceiling for most proteins)

---

### Workflow 2: GPU-Accelerated Simulation

[Step-by-step: system > 5K atoms, GPU nodes available, need 50–200× speedup over CPU]

In [ ]:
%%bash
# WORKFLOW 2: GPU-Accelerated Simulation with pmemd.cuda
# Script: amber_gpu.sbatch  (takes pre-built prmtop + inpcrd as positional args)

module load amber ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -2

echo ""
echo "=== GPU engine check ==="
if command -v pmemd.cuda &>/dev/null; then
    echo "✅ pmemd.cuda is available  (amber module)"
else
    echo "⚠️  pmemd.cuda not found — check: module load amber, and use a GPU node"
fi

echo ""
echo "=== GPU availability ==="
nvidia-smi -L 2>/dev/null || echo "No nvidia-smi (check on gpu partition nodes)"

echo ""
echo "=== Simulation stages ==="
echo "  Stage 1: sander     (minimization — CPU, ambertools)"
echo "  Stage 2: pmemd.cuda (heating 0→300 K — GPU, amber)"
echo "  Stage 3: pmemd.cuda (production NPT, 100 ps — GPU, amber)"

# PRODUCTION SUBMISSION (uncomment when ready)
# ⚠️ Warning: submits a multi-hour GPU job via amber_gpu.sbatch
# Requires: system.prmtop + system.inpcrd already built by tleap
#
# sbatch --partition=gpu --gres=gpu:1 \
#   amber_gpu.sbatch system.prmtop system.inpcrd output_gpu

**Expected output:**
```
GPU engine check:
✅ pmemd.cuda is available  (amber module)

GPU availability:
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-...)
```

Submit using `amber_gpu.sbatch`:
```bash
sbatch --partition=gpu --gres=gpu:1 \
  amber_gpu.sbatch system.prmtop system.inpcrd output_gpu
```

`amber_gpu.sbatch` runs `sander (EM, CPU) → pmemd.cuda (heat) → pmemd.cuda (prod, 100 ps)` and saves results under `output_gpu/`.

**⚠️ Before submitting:**
1. Confirm `system.prmtop` and `system.inpcrd` exist (built by `tleap`)
2. Check GPU partition availability: `sinfo -t available -p gpu`
3. Energy minimization always runs on CPU (`sander`); heat and production on GPU (`pmemd.cuda`)

---

### Workflow 3: Multi-Node Production Run

[Step-by-step: very large systems > 100K atoms, long production runs, multi-node needed]

In [ ]:
%%bash
# WORKFLOW 3: Multi-Node Production Run with sander.MPI
# Script: amber_mpi.sbatch  (takes pre-built prmtop + inpcrd as positional args)

module load amber ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -2

echo ""
echo "=== MPI & Interconnect Info ==="
which mpirun && mpirun --version 2>&1 | head -2

echo ""
echo "=== Simulation stages ==="
echo "  Stage 1: sander     (minimization — serial, ambertools)"
echo "  Stage 2: sander.MPI (heating 0→300 K — multi-node, ambertools)"
echo "  Stage 3: sander.MPI (production NPT, 100 ps — multi-node, ambertools)"

echo ""
echo "=== Cluster resources ==="
sinfo -N 2>/dev/null | head -10 || echo "(sinfo not available)"

# PRODUCTION SUBMISSION (uncomment when ready)
# ⚠️ Warning: submits a multi-hour multi-node job via amber_mpi.sbatch
# Requires: system.prmtop + system.inpcrd already built by tleap
#
# sbatch --nodes=4 --ntasks-per-node=8 \
#   amber_mpi.sbatch system.prmtop system.inpcrd output_mpi

**Expected output:**
```
MPI & Interconnect Info:
/usr/mpi/bin/mpirun
mpirun (Open MPI) 4.x.x ...
```

Submit using `amber_mpi.sbatch`:
```bash
sbatch --partition=build --nodes=4 --ntasks-per-node=8 \
  amber_mpi.sbatch system.prmtop system.inpcrd output_mpi
```

`amber_mpi.sbatch` runs `sander (EM, serial) → sander.MPI (heat + prod)` across all allocated nodes and saves results under `output_mpi/`.

**⚠️ Before submitting:**
1. Confirm `system.prmtop` and `system.inpcrd` exist (built by `tleap`)
2. Match `--ntasks-per-node` to the number of cores per physical node
3. If the job hits its time limit, restart from the last `.rst7` with `irest=1, ntx=5` in the `.mdin`

---

## PART IV — Troubleshooting (Skim When Broken)

**Purpose:** "Something failed. Give me answers fast."

### Monitoring Your Job

In [ ]:
%%bash
echo "=== 1. Check Current Job Status ==="
squeue -u $USER --format="%.7i %.20j %.2t %.10M %.6D %R"

echo ""
echo "=== 2. Get Partition Info ==="
sinfo | grep -E "(gpu|build|cpu)" | head -5

echo ""
echo "=== 3. View Completed Jobs ==="
sacct -u $USER --format=JobID,JobName,State,Elapsed | tail -5

In [ ]:
%%bash
# Comprehensive environment diagnostics
module load amber ambertools 2>/dev/null

echo "=== Amber Environment Diagnostics ==="
echo ""

echo "1. ambertools binaries"
for tool in tleap sander sander.MPI cpptraj; do
    command -v $tool &>/dev/null \
        && echo "  ✅ $tool: $(which $tool)" \
        || echo "  ⚠️  $tool not in PATH — check: module load ambertools"
done
echo ""

echo "2. amber binaries"
command -v pmemd.cuda &>/dev/null \
    && echo "  ✅ pmemd.cuda: $(which pmemd.cuda)" \
    || echo "  ⚠️  pmemd.cuda not in PATH (needs GPU node + module load amber)"
echo ""

echo "3. MPI Library"
command -v mpirun &>/dev/null \
    && echo "✅ MPI: $(mpirun --version 2>&1 | head -1)" \
    || echo "⚠️  MPI not in PATH"
echo ""

echo "4. GPU Detection"
if command -v nvidia-smi &>/dev/null; then
    echo "✅ NVIDIA GPU found:"
    nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader 2>/dev/null
else
    echo "⚠️  No GPU detected (GPU available only on gpu partition)"
fi
echo ""

echo "5. Available Partitions"
sinfo -h --format="%.10P %.5D %.14T %.11l" 2>/dev/null | head -10 || echo "(Partition info not available)"

### Symptom-Based Lookup

| Symptom | Likely Cause | Fix |
|---------|--------------|-----|
| `sander: command not found` | Module not loaded | `module load ambertools` |
| `pmemd.cuda: command not found` | On CPU node or module issue | Submit to `--partition=gpu`; `module load amber` |
| `forrtl: error (78): process killed` | OOM — out of memory | Increase `#SBATCH --mem` or reduce system size |
| `FATAL: Mismatched domain in prmtop` | Wrong topology | Re-run `tleap -f leaprc.input` |
| `CUDA error: out of memory` | System too large for GPU RAM | Reduce atoms or switch to `sander.MPI` |
| `Ewald: sum did not converge` | Unstable starting geometry | Re-run minimization with more `maxcyc` steps |
| `NaN in energy` | Simulation blowing up | Check initial structure; run minimization first |
| Job stuck in `PD` | No resources available | Reduce `--nodes` / `--ntasks` or wait |
| `UCX/MPI abort` | Network issue on multi-node | Add `--mca btl_openib_allow_ib true` to mpirun |

### Understanding Output Files

| File | Description | Check For |
|------|-------------|----------|
| `.prmtop` | Amber topology (from tleap) | Exists and non-zero size |
| `.inpcrd` / `.rst7` | Coordinates / restart file | Final coords for next stage |
| `.mdout` | Simulation energy log | Energies, temperature, pressure vs time |
| `.nc` | NetCDF trajectory | Atomic positions over time (main output) |
| `.mdinfo` | Running summary | Updated each step; shows ns/day performance |

Quick sanity check after any stage:
```bash
# Verify restart file was written
ls -lh *.rst7

# Check energy log tail
tail -20 prod.mdout | grep -E "NSTEP|Etot|Temp|Press|Density"

# Check ns/day performance
grep "ns/day" prod.mdout
```

### FAQ

**Q: What is the difference between `sander` and `pmemd.cuda`?**

A: `sander` is the original CPU-based engine (serial or MPI), included free in AmberTools. `pmemd.cuda` is the GPU-accelerated engine from the licensed Amber package. `pmemd.cuda` is 50–200× faster for proteins > 5K atoms, but requires both an Amber license and a GPU node.

**Q: How do I restart a simulation from a checkpoint?**

A: Use the last `.rst7` file as the new coordinate input. Change your `mdin` to `irest=1, ntx=5`:
```bash
sander -O -i mdin_files/prod.mdin -o prod_cont.mdout \
  -p system.prmtop -c prod.rst7 -r prod_cont.rst7 -x prod_cont.nc
```

**Q: How do I check my simulation's performance (ns/day)?**

A: `grep "ns/day" prod.mdout` or watch live: `tail -f prod.mdinfo | grep Simultime`

**Q: How much disk space do I need?**

A: Rule of thumb: 10× the topology size per stage. A 2 MB `.prmtop` → expect ~20 MB per stage. A 1 μs trajectory (50K atoms, ntwx=500) runs 5–50 GB in NetCDF format.

**Q: Why does my GPU job run slower than expected?**

A: Common causes: (1) System too small (< 5K atoms — GPU overhead dominates). (2) `pmemd.cuda` not in PATH — check `which pmemd.cuda` on the compute node. (3) GPU not being used — run `nvidia-smi` during job to confirm utilization > 90%.

**Q: Can I run multiple proteins at once?**

A: Yes — use `amber.sbatch` with `INPUT_DIR` pointing to a folder of PDB files. It loops over all of them automatically.

**Q: How do I check my job status?**

A: `squeue -u $USER` for running/queued jobs. `sacct -u $USER` for completed jobs.

### External Resources

- 📖 **Amber Reference Manual**: https://ambermd.org/doc12/Amber25.pdf
- 🎓 **Official Tutorials**: https://ambermd.org/tutorials/
- 💻 **Amber Website**: https://ambermd.org/
- 📧 **Mailing List**: https://lists.ambermd.org/mailman/listinfo/amber
- 📊 **cpptraj Analysis Guide**: https://amberhub.chpc.utah.edu/cpptraj-tutorials/